### Intro
- Implementation of RNN, GRU and LSTM from scratch, using pytorch
- PyTorch (pre-0.4), to track gradients, you had to wrap tensors in Variable. fastai=0.7 wraps pytorch Variable with a helper to V.
- pytorch(0.4) deprecated Variable and merged with tensor

In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

from fastai.io import*
from fastai.conv_learner import*

from fastai.column_data import*

### Set up

##### Cuda device setup

In [ ]:
import torch, torchvision
print("torch:", torch.__version__)
print("CUDA toolkit:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
print("torchvision:", torchvision.__version__)
print(torch.backends.cudnn.version())

torch.cuda.set_device(0)
torch.backends.cudnn.enabled = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current CUDA device:", torch.cuda.current_device())
    print("CUDA device name:", torch.cuda.get_device_name(0))

In [2]:
PATH = 'data/nietzsche/'

In [3]:
get_data("https://s3.amazonaws.com/text-datasets/nietzsche.txt", f'{PATH}nietzsche.txt')
text = open(f'{PATH}nietzsche.txt').read()
print('corpus length:', len(text))

corpus length: 600893


In [4]:
text[:400]

'PREFACE\n\n\nSUPPOSING that Truth is a woman--what then? Is there not ground\nfor suspecting that all philosophers, in so far as they have been\ndogmatists, have failed to understand women--that the terrible\nseriousness and clumsy importunity with which they have usually paid\ntheir addresses to Truth, have been unskilled and unseemly methods for\nwinning a woman? Certainly she has never allowed herself '

In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)+1
print('total chars: ', vocab_size)

total chars:  85


- add "\0" as a padding char at the start

In [6]:
chars.insert(0, "\0")

In [7]:
print(chars)

['\x00', '\n', ' ', '!', '"', "'", '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '=', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'Æ', 'ä', 'æ', 'é', 'ë']


In [8]:
''.join(chars[1:-6])

'\n !"\'(),-.0123456789:;=?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxy'

Map from chars to indices and back indices to chars

In [9]:
char_indices = dict((c, i) for i, c in enumerate(chars))
indices_char = dict((i, c) for i, c in enumerate(chars))

In [10]:
idx = [char_indices[c] for c in text]

In [11]:
idx[:10]

[40, 42, 29, 30, 25, 27, 29, 1, 1, 1]

In [12]:
#reverse checking those index indices gives the characters
''.join(indices_char[i] for i in idx[:70])

'PREFACE\n\n\nSUPPOSING that Truth is a woman--what then? Is there not gro'

##### 2 Three Character Model
##### 2.1 Create inputs

Create a list of every 4th charc, starting at 0th,1st,2nd,then 3rd charct

In [13]:
cs=3
c1_dat = [idx[i] for i in range(0, len(idx)-1-cs, cs)]
c2_dat = [idx[i+1] for i in range(0, len(idx)-1-cs, cs)]
c3_dat = [idx[i+2] for i in range(0, len(idx)-1-cs, cs)]
c4_dat = [idx[i+3] for i in range(0, len(idx)-1-cs, cs)]

In [14]:
# print(''.join(indices_char[i] for i in c1_dat[:10]))
# print(''.join(indices_char[i] for i in c2_dat[:10]))
# print(''.join(indices_char[i] for i in c3_dat[:10]))
# print(''.join(indices_char[i] for i in c4_dat[:10]))

In [15]:
x1 = np.stack(c1_dat[:-2])
x2 = np.stack(c2_dat[:-2])
x3 = np.stack(c3_dat[:-2])

In [16]:
y = np.stack(c4_dat[:-2])

In [17]:
x1[:4],x2[:4],x3[:4]

(array([40, 30, 29,  1]), array([42, 25,  1, 43]), array([29, 27,  1, 45]))

In [18]:
y[:4]

array([30, 29,  1, 40])

In [19]:
x1.shape, y.shape

((200295,), (200295,))

##### Create and train model

In [20]:
n_hidden = 256

In [21]:
n_fac = 42

In [22]:
class Char3Model(nn.Module):
    def __init__(self, vocab_size, n_fac):
        super().__init__()
        self.e = nn.Embedding(vocab_size, n_fac)
        self.l_in = nn.Linear(n_fac, n_hidden)
        self.l_hidden = nn.Linear(n_hidden, n_hidden)
        self.l_out = nn.Linear(n_hidden, vocab_size)

    def forward(self, c1, c2, c3):
        in1 = F.relu(self.l_in(self.e(c1)))
        in2 = F.relu(self.l_in(self.e(c2)))
        in3 = F.relu(self.l_in(self.e(c3)))

        #make h matrix int zero for refactoring(make them identical) the next lines into a loop
        h = V(torch.zeros(in1.size()))#.cuda()) 
        h = F.tanh(self.l_hidden(h+in1))
        h = F.tanh(self.l_hidden(h+in2))
        h = F.tanh(self.l_hidden(h+in3))

        return F.log_softmax(self.l_out(h))

In [23]:
md = ColumnarModelData.from_arrays('.', [-1], np.stack([x1,x2,x3], axis=1), y, bs=512)

In [24]:
m = Char3Model(vocab_size, n_fac)#.cuda()

In [25]:
it = iter(md.trn_dl)
*xs, yt = next(it)
t = m(*V(xs))

In [26]:
t

Variable containing:
-4.4804 -4.3677 -4.5530  ...  -4.5014 -4.5452 -4.5368
-4.5479 -4.3567 -4.5348  ...  -4.4958 -4.6127 -4.4810
-4.4285 -4.4998 -4.7124  ...  -4.9076 -4.6991 -4.2167
          ...             ⋱             ...          
-4.2850 -4.4469 -4.6333  ...  -4.6539 -4.7375 -4.5470
-4.4259 -4.5127 -4.6060  ...  -4.4909 -4.5350 -4.6146
-4.6317 -4.5703 -4.6195  ...  -4.4702 -4.7942 -4.3618
[torch.FloatTensor of size 512x85]

In [27]:
opt = optim.Adam(m.parameters(), 1e-2)

In [28]:
fit(m, md, 1, opt, F.nll_loss)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      2.085385   6.743257  


[array([6.74326])]

setting learning rate manually as fastai learner, lr finder is not used 

In [29]:
set_lrs(opt, 0.001)

In [30]:
fit(m, md, 1, opt, F.nll_loss)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.836973   5.928057  


[array([5.92806])]

##### Test Model

In [31]:
def get_next(inp):
    idxs = T(np.array([char_indices[c] for c in inp]))
    p = m(*VV(idxs))
    i = np.argmax(to_np(p))
    return chars[i]

In [32]:
get_next('y. ')

'T'

In [33]:
get_next('ppl')

'e'

In [34]:
get_next(' th')

'e'

##### 3 First RNN
###### 3.1 Create inputs

In [28]:
cs = 8

In [29]:
c_in_dat = [[idx[i+j] for i in range(cs)] for j in range(len(idx)-cs-1)]

In [30]:
c_out_dat = [idx[j+cs] for j in range(len(idx)-cs-1)]

In [31]:
xs = np.stack(c_in_dat, axis=0)

In [32]:
xs.shape

(600884, 8)

In [33]:
y = np.stack(c_out_dat)

In [34]:
#overlapping set of 8 characters, the 1st 8th, 2nd 8th char.....
xs[:cs, :cs]

array([[40, 42, 29, 30, 25, 27, 29,  1],
       [42, 29, 30, 25, 27, 29,  1,  1],
       [29, 30, 25, 27, 29,  1,  1,  1],
       [30, 25, 27, 29,  1,  1,  1, 43],
       [25, 27, 29,  1,  1,  1, 43, 45],
       [27, 29,  1,  1,  1, 43, 45, 40],
       [29,  1,  1,  1, 43, 45, 40, 40],
       [ 1,  1,  1, 43, 45, 40, 40, 39]])

In [35]:
y[:cs]

array([ 1,  1, 43, 45, 40, 40, 39, 43])

#### Create and train model

In [36]:
val_idx = get_cv_idxs(len(idx)-cs-1)

In [37]:
md = ColumnarModelData.from_arrays('.', val_idx, xs, y, bs=512)

In [38]:
class CharLoopModel(nn.Module):
    def __init__(self, vocab_size, n_fac):
        super().__init__()
        self.e = nn.Embedding(vocab_size, n_fac)
        self.l_in = nn.Linear(n_fac, n_hidden)
        self.l_hidden = nn.Linear(n_hidden, n_hidden)
        self.l_out = nn.Linear(n_hidden, vocab_size)

    def forward(self, *cs):
        bs = cs[0].size(0)
        #print("bs size is:", bs)
        h = V(torch.zeros(bs, n_hidden))#.cuda())
        for c in cs:
            inp = F.relu(self.l_in(self.e(c)))
            h = F.tanh(self.l_hidden(h+inp))

        return F.log_softmax(self.l_out(h))

In [39]:
m = CharLoopModel(vocab_size, n_fac)#.cuda()
opt = optim.Adam(m.parameters(), 1e-2)

In [47]:
fit(m, md, 1, opt, F.nll_loss)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      2.035542   2.032291  


[array([2.03229])]

In [48]:
set_lrs(opt, 0.001)

In [49]:
fit(m, md, 1, opt, F.nll_loss)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.762037   1.758503  


[array([1.7585])]

In [40]:
class CharLoopConcatModel(nn.Module):
    def __init__(self, vocab_size, n_fac):
        super().__init__()
        self.e = nn.Embedding(vocab_size, n_fac)
        self.l_in = nn.Linear(n_fac+n_hidden, n_hidden)
        self.l_hidden = nn.Linear(n_hidden, n_hidden)
        self.l_out = nn.Linear(n_hidden, vocab_size)

    def forward(self, *cs):
        bs = cs[0].size(0)
        h = V(torch.zeros(bs, n_hidden))#.cuda()
        for c in cs:
            inp = torch.cat((h, self.e(c)), 1)
            inp = F.relu(self.l_in(inp))
            h = F.tanh(self.l_hidden(inp))

        return F.log_softmax(self.l_out(h))

In [41]:
m = CharLoopConcatModel(vocab_size, n_fac)#.cuda()
opt = optim.Adam(m.parameters(), 1e-3)

In [42]:
it = iter(md.trn_dl)
*xs, yt = next(it)
t = m(*V(xs))

In [53]:
fit(m, md, 1, opt, F.nll_loss)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.85237    1.818754  


[array([1.81875])]

In [54]:
set_lrs(opt,1e-4)

In [55]:
fit(m, md, 1, opt, F.nll_loss)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.748169   1.73892   


[array([1.73892])]

##### Test Model

In [56]:
def get_next(inp):
    idxs = T(np.array([char_indices[c] for c in inp]))
    p = m(*VV(idxs))
    i = np.argmax(to_np(p))
    return chars[i]

In [57]:
get_next('for thos')

'e'

In [60]:
get_next('my birth')

'e'

In [61]:
get_next('translat')

'i'

#### RNN with pytorch

In [43]:
class CharRnn(nn.Module):
    def __init__(self, vocab_size, n_fac):
        super().__init__()
        self.e = nn.Embedding(vocab_size, n_fac)
        self.rnn = nn.RNN(n_fac, n_hidden)
        self.l_out = nn.Linear(n_hidden, vocab_size)
    def forward(self, *cs):
        bs = cs[0].size(0)
        h = V(torch.zeros(1, bs, n_hidden))
        inp = self.e(torch.stack(cs))
        outp, h = self.rnn(inp, h)

        #pytorch rnn gives all the hidden state h stacked, get the last one for softmax
        return F.log_softmax(self.l_out(outp[-1]))

In [44]:
m = CharRnn(vocab_size, n_fac)#.cuda()
opt = optim.Adam(m.parameters(), 1e-3)

In [45]:
it = iter(md.trn_dl)
*xs, yt = next(it)

In [46]:
t = m.e(V(torch.stack(xs)))
t.size()

torch.Size([8, 512, 42])

In [47]:
ht = V(torch.zeros(1, 512, n_hidden))
outp, hn = m.rnn(t, ht)
outp.size(), hn.size()

(torch.Size([8, 512, 256]), torch.Size([1, 512, 256]))

In [48]:
t = m(*V(xs)); t.size()

torch.Size([512, 85])

In [68]:
fit(m, md, 4, opt, F.nll_loss)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.87112    1.842104  
    1      1.685934   1.674251                              
    2      1.588082   1.594748                              
    3      1.532305   1.556772                              


[array([1.55677])]

In [69]:
set_lrs(opt, 1e-4)

In [70]:
fit(m, md, 2, opt, F.nll_loss)

Epoch:   0%|          | 0/2 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.464697   1.512981  
    1      1.458336   1.507377                              


[array([1.50738])]

##### Test Model

In [71]:
def get_next(inp):
    idxs = T(np.array([char_indices[c] for c in inp]))
    p = m(*VV(idxs))
    i = np.argmax(to_np(p))
    return chars[i]

In [72]:
get_next('for thos')

'e'

In [75]:
def get_next_n(inp, n):
    res = inp
    for i in range(n):
        c = get_next(inp)
        res += c
        inp = inp[1:]+c
    return res

In [77]:
get_next_n('for thos', 45)

'for those of the same and the strength and the streng'

### Multi-output model
###### set up
lets take non-overlapping sets of characters

In [49]:
c_in_dat = [[idx[i+j] for i in range(cs)] for j in range(0, len(idx)-cs-1, cs)]

Then creat the exact same thing, offset by 1, as our labels

In [50]:
c_out_dat = [[idx[i+j] for i in range(cs)] for j in range(1, len(idx)-cs, cs)]

In [51]:
xs = np.stack(c_in_dat)
xs.shape

(75111, 8)

In [52]:
ys = np.stack(c_out_dat)
ys.shape

(75111, 8)

In [53]:
xs[:cs, :cs]

array([[40, 42, 29, 30, 25, 27, 29,  1],
       [ 1,  1, 43, 45, 40, 40, 39, 43],
       [33, 38, 31,  2, 73, 61, 54, 73],
       [ 2, 44, 71, 74, 73, 61,  2, 62],
       [72,  2, 54,  2, 76, 68, 66, 54],
       [67,  9,  9, 76, 61, 54, 73,  2],
       [73, 61, 58, 67, 24,  2, 33, 72],
       [ 2, 73, 61, 58, 71, 58,  2, 67]])

In [54]:
ys[:cs, :cs]

array([[42, 29, 30, 25, 27, 29,  1,  1],
       [ 1, 43, 45, 40, 40, 39, 43, 33],
       [38, 31,  2, 73, 61, 54, 73,  2],
       [44, 71, 74, 73, 61,  2, 62, 72],
       [ 2, 54,  2, 76, 68, 66, 54, 67],
       [ 9,  9, 76, 61, 54, 73,  2, 73],
       [61, 58, 67, 24,  2, 33, 72,  2],
       [73, 61, 58, 71, 58,  2, 67, 68]])

#### Create and train model

In [55]:
val_idx = get_cv_idxs(len(xs)-cs-1)

In [56]:
md = ColumnarModelData.from_arrays('.', val_idx, xs, ys, bs=512)

In [57]:
class CharSeqRnn(nn.Module):
    def __init__(self, vocab_size, n_fac):
        super().__init__()
        self.e = nn.Embedding(vocab_size, n_fac)
        self.rnn = nn.RNN(n_fac, n_hidden)
        self.l_out = nn.Linear(n_hidden, vocab_size)

    def forward(self, *cs):
        bs = cs[0].size(0)
        h = V(torch.zeros(1, bs, n_hidden))
        inp = self.e(torch.stack(cs))
        outp, h = self.rnn(inp, h)
        
        return F.log_softmax(self.l_out(outp), dim=-1)

In [58]:
m = CharSeqRnn(vocab_size, n_fac)#.cuda()
opt = optim.Adam(m.parameters(), 1e-3)

In [59]:
it = iter(md.trn_dl)
*xst, yt = next(it)

In [60]:
def nll_loss_seq(inp, targ):
    sl, bs, nh = inp.size()
    targ = targ.transpose(0,1).contiguous().view(-1)
    return F.nll_loss(inp.view(-1, nh), targ)


In [63]:
fit(m, md, 4, opt, nll_loss_seq)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      2.595383   2.412234  
    1      2.29451    2.20883                               
    2      2.144946   2.094292                              
    3      2.050442   2.014885                              


[array([2.01489])]

In [64]:
 set_lrs(opt, 1e-4)

In [65]:
fit(m, md, 1, opt, nll_loss_seq)

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                             
    0      1.999122   2.001164  


[array([2.00116])]

#### indentity initiazation

In [61]:
m = CharSeqRnn(vocab_size, n_fac)#.cuda()
opt = optim.Adam(m.parameters(), 1e-2)

## jeff hinton implementation paper
(A Simple Way to Initialize Recurrent Networks of Rectified Linear Units)
to initialize the weights of the hidden layer with identity matrix 

In [62]:
m.rnn.weight_hh_l0.data.copy_(torch.eye(n_hidden))


    1     0     0  ...      0     0     0
    0     1     0  ...      0     0     0
    0     0     1  ...      0     0     0
       ...          ⋱          ...       
    0     0     0  ...      1     0     0
    0     0     0  ...      0     1     0
    0     0     0  ...      0     0     1
[torch.FloatTensor of size 256x256]

In [64]:
fit(m, md, 4, opt, nll_loss_seq)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      2.213493   2.114454  
    1      2.033242   1.982409                              
    2      1.949431   1.915228                              
    3      1.902204   1.892601                              


[array([1.8926])]

In [ ]:
set_lrs(opt, 1e-3)

In [71]:
fit(m, md, 4, opt, nll_loss_seq)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.823682   1.840258  
    1      1.809141   1.833065                              
    2      1.80187    1.827229                             
    3      1.790729   1.822655                              


[array([1.82266])]

In [72]:
set_lrs(opt, 1e-4)

In [73]:
fit(m, md, 4, opt, nll_loss_seq)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.780542   1.81642   
    1      1.778972   1.815632                              
    2      1.777121   1.81507                               
    3      1.777158   1.814402                              


[array([1.8144])]

##### Stateful 

setup

In [65]:
from torchtext import vocab, data

from fastai.nlp import*
from fastai.lm_rnn import *

PATH = ('data/nietzsche/')

TRN_PATH = 'trn/'
VAL_PATH = 'val/'

# TRN_PATH.mkdir(exist_ok=True)
# VAL_PATH.mkdir(exist_ok=True)

TRN = f'{PATH}{TRN_PATH}'
VAL = f'{PATH}{VAL_PATH}'

%ls {PATH}

nietzsche.txt*  trn/  val/


In [66]:
%ls {PATH}trn/

trn.txt*


In [67]:
with open(f'{PATH}nietzsche.txt' , 'r', encoding='utf-8') as f:
    text = f.read()

split = int(len(text) * 0.83529)
trn_txt = text[:split]
val_txt = text[split:]

with open(f'{PATH}trn/trn.txt' , 'w', encoding='utf-8') as f:
    f.write(trn_txt)

with open(f'{PATH}val/val.txt' , 'w', encoding='utf-8') as f:
    f.write(val_txt)

In [68]:
TEXT = data.Field(lower=True, tokenize=list)
bs=64; bptt=8; n_fac=42; n_hidden=256

FILES = dict(train=TRN_PATH, validation=VAL_PATH, test=VAL_PATH)

md = LanguageModelData.from_text_files(PATH, TEXT, **FILES, bs=bs, bptt=bptt, min_freq=2)

len(md.trn_dl), md.nt, len(md.trn_ds), len(md.trn_ds[0].text)

(963, 56, 1, 493745)

#### RNN

##### Backpropagation through time
After your for loop(e.g this case 8) remember the tensor data but through away history of operatioins and start fresh
keep our hidden state but not the hidden state history.

so that during backpropagation it stops there. How many layers to backprop. 
- Best not to BPTT many layers, if you have gradient instability i.e. (explosion or vanishing) the harder to train
- but longer BPTT means able to explicitly capture longer memory or state  
here repackage_var(h) is doing that

In [69]:
class CharSeqStatefulRnn(nn.Module):
    def __init__(self, vocab_size, n_fac, bs):
        self.vocab_size = vocab_size
        super().__init__()
        self.e = nn.Embedding(vocab_size, n_fac)
        self.rnn = nn.RNN(n_fac, n_hidden)
        self.l_out = nn.Linear(n_hidden, vocab_size)
        self.init_hidden(bs)

    def forward(self, cs):
        bs = cs[0].size(0)
        if self.h.size(1) != bs: self.init_hidden(bs)
        outp, h = self.rnn(self.e(cs), self.h)
        #Backpropagation through time(BPTT)
        self.h = repackage_var(h) # remember the state with out the calculation/operations history, by just taking the tensor data and create new Variable 
        return F.log_softmax(self.l_out(outp), dim=-1).view(-1, self.vocab_size)

    def init_hidden(self, bs): self.h = V(torch.zeros(1, bs, n_hidden))

In [70]:
m = CharSeqStatefulRnn(md.nt, n_fac, 512)#.cuda()
opt = optim.Adam(m.parameters(), 1e-3)

In [71]:
fit(m, md, 4, opt, F.nll_loss)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

epoch      trn_loss   val_loss                              
    0      1.885144   1.852106  
    1      1.702605   1.696594                              
    2      1.621045   1.630495                              
    3      1.566336   1.58942                               


[array([1.58942])]